# Lab 2 — Sequence labelling, constraints and beam search

[Open this notebook in Colab](https://colab.research.google.com/github/coastalcph/nlp-course/blob/master/labs/notebooks_2026/lab_2.ipynb), then select **File → Save a copy in Drive**.

This lab treats sequence labelling as a task: define labels, build an interpretable token classifier, evaluate complete spans and enforce valid BIO outputs. RNN and Transformer architectures are covered later.

## Learning objectives

By the end of this lab, you should be able to:

1. formulate span prediction using BIO-style sequence labels;
2. train and evaluate a feature-based token classifier;
3. compare independent prediction, BIO repair and constrained beam search; and
4. map character-level answer spans to token-level project labels and back.

## TA session plan

1. Inspect the named-entity data, label distribution and BIO encoding.
2. Train the token-level baseline and compare token and span metrics.
3. Diagnose invalid transitions and run constrained beam search.
4. Test character-span conversion on project-style examples with a TA.

# Setup

The lab uses a small linear classifier. No recurrent or Transformer model is required.

In [1]:
%pip install -q "datasets>=4,<5" "pandas>=2.2,<3" "scikit-learn>=1.6,<2"

Note: you may need to restart the kernel to use updated packages.


In [2]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [12]:
import random
import re

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

random.seed(42)
np.random.seed(42)

# Sequence labelling is an aligned prediction task

Sequence labelling assigns one label to each chosen input unit. Examples include:

* part-of-speech tagging;
* named-entity recognition;
* propaganda-technique spans; and
* answer-span prediction in extractive question answering.

The task definition does not prescribe an RNN, Transformer or any other architecture.

## Named-entity recognition

Consider the sentence:

| Sundar | Pichai | leads | Alphabet | in | California | . |
|-|-|-|-|-|-|-|
| PER | PER | O | ORG | O | LOC | O |

This identifies entity types but does not mark where a multi-token entity begins.

## BIO encoding adds boundaries

| Sundar | Pichai | leads | Alphabet | in | California | . |
|-|-|-|-|-|-|-|
| B-PER | I-PER | O | B-ORG | O | B-LOC | O |

BIO labels also define constraints. For example, `I-PER` cannot start a sequence or directly follow `B-LOC`.

# Download and inspect CoNLL-2003

We use English news text labelled with people, locations, organisations and miscellaneous entities.

In [4]:
CONLL_PARQUET = "https://huggingface.co/datasets/eriktks/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003"
data_files = {
    split: f"{CONLL_PARQUET}/{split}/0000.parquet"
    for split in ("train", "validation", "test")
}
datasets = load_dataset("parquet", data_files=data_files)

NER_TAGS = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC",
]
TAG_TO_ID = {tag: index for index, tag in enumerate(NER_TAGS)}
datasets

Generating train split: 14041 examples [00:00, 314699.43 examples/s]
Generating validation split: 3250 examples [00:00, 300081.19 examples/s]
Generating test split: 3453 examples [00:00, 435655.51 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [5]:
example = datasets["train"][0]
pd.DataFrame({
    "token": example["tokens"],
    "label": [NER_TAGS[label] for label in example["ner_tags"]],
})

,token,label
0,EU,B-ORG
1,rejects,O
2,German,B-MISC
3,call,O
4,to,O
5,boycott,O
6,British,B-MISC
7,lamb,O
8,.,O


## Audit the label distribution

Token accuracy can be misleading because most tokens are outside any entity. Always compare with the `O` baseline and report entity-sensitive metrics.

In [6]:
train_label_counts = pd.Series(
    [label for example in datasets["train"] for label in example["ner_tags"]]
).value_counts().reindex(range(len(NER_TAGS)), fill_value=0)

pd.DataFrame({
    "label": NER_TAGS,
    "count": train_label_counts.values,
    "proportion": train_label_counts.values / train_label_counts.sum(),
})

,label,count,proportion
0,O,169578,0.832812
1,B-PER,6600,0.032413
2,I-PER,4528,0.022237
3,B-ORG,6321,0.031043
4,I-ORG,3704,0.018191
5,B-LOC,7140,0.035065
6,I-LOC,1157,0.005682
7,B-MISC,3438,0.016884
8,I-MISC,1155,0.005672


# Evaluate tokens and complete spans

For NER, a predicted entity is correct only when its type and both boundaries match. The helper below reports token accuracy, macro-F1 over entity labels and exact-span precision/recall/F1.

In [7]:
def bio_spans(sequence):
    spans = set()
    active_type = None
    start = None

    for index, label_id in enumerate(list(sequence) + [TAG_TO_ID["O"]]):
        label = NER_TAGS[int(label_id)]
        if label == "O":
            prefix, entity_type = "O", None
        else:
            prefix, entity_type = label.split("-", 1)

        if prefix == "I" and entity_type == active_type:
            continue

        if active_type is not None:
            spans.add((start, index, active_type))
            active_type = None
            start = None

        if prefix in {"B", "I"}:
            active_type = entity_type
            start = index

    return spans


def span_scores(gold_sequences, predicted_sequences):
    predicted_count = gold_count = matches = 0
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        gold_spans = bio_spans(gold)
        predicted_spans = bio_spans(predicted)
        gold_count += len(gold_spans)
        predicted_count += len(predicted_spans)
        matches += len(gold_spans & predicted_spans)

    precision = matches / predicted_count if predicted_count else 0.0
    recall = matches / gold_count if gold_count else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def evaluate_sequences(name, gold_sequences, predicted_sequences):
    gold_flat = np.concatenate(gold_sequences)
    predicted_flat = np.concatenate(predicted_sequences)
    precision, recall, span_f1 = span_scores(gold_sequences, predicted_sequences)
    result = {
        "model": name,
        "token_accuracy": accuracy_score(gold_flat, predicted_flat),
        "entity_token_macro_f1": f1_score(
            gold_flat,
            predicted_flat,
            labels=list(range(1, len(NER_TAGS))),
            average="macro",
            zero_division=0,
        ),
        "span_precision": precision,
        "span_recall": recall,
        "span_f1": span_f1,
    }
    return result


def label_sequences(split):
    return [list(example["ner_tags"]) for example in datasets[split]]

## Baseline 1: always predict outside

This baseline may obtain high token accuracy while finding no entities.

In [8]:
validation_gold = label_sequences("validation")
only_o_predictions = [
    [TAG_TO_ID["O"]] * len(sequence) for sequence in validation_gold
]
baseline_scores = evaluate_sequences(
    "Always O", validation_gold, only_o_predictions
)
pd.DataFrame([baseline_scores])

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Always O,0.832503,0.0,0.0,0.0,0.0


# Baseline 2: classify each token

Represent each token with transparent features. Context comes from neighbouring words rather than a recurrent architecture.

In [11]:
def token_features(tokens, index):
    word = tokens[index]
    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    return features


def featurize_split(split, max_sentences=None):
    data = datasets[split]
    if max_sentences is not None:
        data = data.select(range(min(max_sentences, len(data))))

    features, labels, lengths = [], [], []
    for example in data:
        tokens = example["tokens"]
        lengths.append(len(tokens))
        features.extend(token_features(tokens, index) for index in range(len(tokens)))
        labels.extend(example["ner_tags"])
    return features, np.asarray(labels), lengths


def split_by_lengths(values, lengths):
    sequences = []
    offset = 0
    for length in lengths:
        sequences.append(list(values[offset:offset + length]))
        offset += length
    assert offset == len(values)
    return sequences

In [15]:
TRAIN_SENTENCES = 4000

train_features, train_labels, train_lengths = featurize_split(
    "train", max_sentences=TRAIN_SENTENCES
)
validation_features, validation_labels, validation_lengths = featurize_split(
    "validation"
)

vectorizer = DictVectorizer(sparse=True)
X_train = vectorizer.fit_transform(train_features)
X_validation = vectorizer.transform(validation_features)
X_train.indices = X_train.indices.astype(np.int32) #added
X_train.indptr = X_train.indptr.astype(np.int32)   #added

token_classifier = SGDClassifier(
    loss="log_loss",
    alpha=1e-5,
    class_weight="balanced",
    max_iter=30,
    random_state=42,
)
token_classifier.fit(X_train, train_labels)

independent_flat = token_classifier.predict(X_validation)
independent_predictions = split_by_lengths(
    independent_flat, validation_lengths
)
independent_scores = evaluate_sequences(
    "Independent token classifier", validation_gold, independent_predictions
)

pd.DataFrame([baseline_scores, independent_scores])

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Always O,0.832503,0.000000,0.000000,0.000000,0.000000
1,Independent token classifier,0.957770,0.761261,0.699455,0.777853,0.736574


# Error analysis

Inspect boundary and type errors. Which missing feature would address a recurring error without referring to the validation labels directly?

In [16]:
confusion = confusion_matrix(
    np.concatenate(validation_gold),
    independent_flat,
    labels=list(range(len(NER_TAGS))),
)
pd.DataFrame(confusion, index=NER_TAGS, columns=NER_TAGS)

,O,B-PER,I-PER,B-ORG,I-ORG,B-LOC,I-LOC,B-MISC,I-MISC
O,42330,77,24,132,83,54,2,30,27
B-PER,56,1548,23,93,14,83,0,24,1
I-PER,33,5,1210,8,28,8,3,5,7
B-ORG,49,127,9,980,32,104,2,35,3
I-ORG,60,14,73,21,515,33,14,4,17
B-LOC,47,53,4,130,10,1561,0,32,0
I-LOC,19,0,31,3,28,9,155,0,12
B-MISC,65,38,4,36,14,62,0,689,14
I-MISC,46,1,26,2,25,8,9,24,205


In [17]:
error_rows = []
for example, gold, predicted in zip(
    datasets["validation"], validation_gold, independent_predictions
):
    for index, (gold_id, predicted_id) in enumerate(zip(gold, predicted)):
        if gold_id != predicted_id:
            error_rows.append({
                "context": " ".join(example["tokens"]),
                "token": example["tokens"][index],
                "gold": NER_TAGS[gold_id],
                "predicted": NER_TAGS[int(predicted_id)],
            })

pd.DataFrame(error_rows).head(20)

,context,token,gold,predicted
0,West Indian all-rounder Phil Simmons took four...,West,B-MISC,B-LOC
1,West Indian all-rounder Phil Simmons took four...,Indian,I-MISC,I-LOC
2,After bowling Somerset out for 83 on the openi...,Road,I-LOC,I-PER
3,"Trailing by 213 , Somerset got a solid start t...",Simmons,B-PER,O
4,"Essex , however , look certain to regain their...",Hussain,I-PER,I-ORG
5,"Hussain , considered surplus to England 's one...",Hussain,B-PER,B-LOC
6,By the close Yorkshire had turned that into a ...,Such,B-PER,B-ORG
7,Australian Tom Moody took six for 82 but Chris...,Tom,B-PER,I-ORG
8,They were held up by a gritty 84 from Paul Joh...,ex-England,B-MISC,O
9,They were held up by a gritty 84 from Paul Joh...,McCague,I-PER,O


# BIO constraints

Independent predictions can produce impossible transitions. A lightweight repair converts an illegal `I-X` into `B-X`; it changes decoding, not the token classifier.

In [18]:
def valid_bio_transition(previous_id, current_id):
    current = NER_TAGS[int(current_id)]
    if not current.startswith("I-"):
        return True
    if previous_id is None:
        return False
    entity_type = current[2:]
    return NER_TAGS[int(previous_id)] in {f"B-{entity_type}", f"I-{entity_type}"}


def repair_bio(sequence):
    repaired = []
    for label_id in sequence:
        if valid_bio_transition(repaired[-1] if repaired else None, label_id):
            repaired.append(int(label_id))
        else:
            repaired.append(TAG_TO_ID["B-" + NER_TAGS[int(label_id)][2:]])
    return repaired


repaired_predictions = [repair_bio(sequence) for sequence in independent_predictions]
repaired_scores = evaluate_sequences(
    "Independent + BIO repair", validation_gold, repaired_predictions
)
pd.DataFrame([baseline_scores, independent_scores, repaired_scores])

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Always O,0.832503,0.000000,0.000000,0.000000,0.000000
1,Independent token classifier,0.957770,0.761261,0.699455,0.777853,0.736574
2,Independent + BIO repair,0.954889,0.744450,0.699455,0.777853,0.736574


# Constrained beam search

Repair acts only after greedy prediction. Beam search instead compares several partial label sequences while blocking invalid BIO transitions. The encoder remains the same feature-based token classifier.

In [19]:
def constrained_beam_decode(token_log_probs, class_ids, beam_size=4):
    beam = [([], 0.0)]

    for scores in token_log_probs:
        candidates = []
        for sequence, sequence_score in beam:
            previous = sequence[-1] if sequence else None
            for column, label_id in enumerate(class_ids):
                if valid_bio_transition(previous, label_id):
                    candidates.append((
                        sequence + [int(label_id)],
                        sequence_score + float(scores[column]),
                    ))
        candidates.sort(key=lambda item: item[1], reverse=True)
        beam = candidates[:beam_size]

    return beam[0][0]


validation_log_probs = token_classifier.predict_log_proba(X_validation)
log_prob_sequences = split_by_lengths(validation_log_probs, validation_lengths)
beam_predictions = [
    constrained_beam_decode(
        scores, token_classifier.classes_, beam_size=4
    )
    for scores in log_prob_sequences
]
beam_scores = evaluate_sequences(
    "Constrained beam (4)", validation_gold, beam_predictions
)

pd.DataFrame([
    baseline_scores,
    independent_scores,
    repaired_scores,
    beam_scores,
]).sort_values("span_f1", ascending=False)

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
3,Constrained beam (4),0.960243,0.775191,0.762723,0.799562,0.780708
1,Independent token classifier,0.957770,0.761261,0.699455,0.777853,0.736574
2,Independent + BIO repair,0.954889,0.744450,0.699455,0.777853,0.736574
0,Always O,0.832503,0.000000,0.000000,0.000000,0.000000


## Interpret the comparison

1. How often did the independent classifier produce invalid BIO transitions?
2. Did repair and beam search change token accuracy and span-F1 in the same direction?
3. Why can a beam larger than one help even though the per-token scores are fixed?
4. What transition information might a learned structured model add later?

# Project transfer: character spans to BIO labels

The project stores answer spans as character offsets in an English context. A sequence labeller needs token-level labels, so the conversion must be explicit and reversible.

In [20]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets


def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [21]:
context = "Ada Lovelace wrote the first algorithm."
answer_text = "Ada Lovelace"
answer_start = context.index(answer_text)

tokens, offsets, labels = character_span_to_bio(
    context, answer_start, answer_text
)
round_trip_start, round_trip_text = bio_to_character_span(
    context, offsets, labels
)

assert answer_start == 0
assert round_trip_start == answer_start
assert round_trip_text == answer_text
assert labels[:2] == ["B-ANS", "I-ANS"]

pd.DataFrame({"token": tokens, "offset": offsets, "label": labels})

,token,offset,label
0,Ada,"(0, 3)",B-ANS
1,Lovelace,"(4, 12)",I-ANS
2,wrote,"(13, 18)",O
3,the,"(19, 22)",O
4,first,"(23, 28)",O
5,algorithm,"(29, 38)",O
6,.,"(38, 39)",O


In [22]:
unanswerable_tokens, unanswerable_offsets, unanswerable_labels = character_span_to_bio(
    context, None, ""
)
assert set(unanswerable_labels) == {"O"}

second_context = "The event was on 1 July 2000."
second_answer = "1 July 2000"
second_start = second_context.index(second_answer)
second_tokens, second_offsets, second_labels = character_span_to_bio(
    second_context, second_start, second_answer
)
assert bio_to_character_span(
    second_context, second_offsets, second_labels
) == (second_start, second_answer)

## Apply the conversion to project data

For at least five answerable and five unanswerable examples:

1. convert the stored character span to BIO labels;
2. convert those labels back to a character span;
3. assert exact recovery of the original answer; and
4. inspect any failure before training a model.

# Evaluate the frozen pipeline once on test data

After choosing features and decoding on validation data, apply the unchanged vectorizer, classifier and beam size to the test split.

In [23]:
test_features, test_labels, test_lengths = featurize_split("test")
X_test = vectorizer.transform(test_features)
test_log_probs = token_classifier.predict_log_proba(X_test)
test_log_prob_sequences = split_by_lengths(test_log_probs, test_lengths)
test_predictions = [
    constrained_beam_decode(
        scores, token_classifier.classes_, beam_size=4
    )
    for scores in test_log_prob_sequences
]
test_gold = label_sequences("test")

pd.DataFrame([
    evaluate_sequences("Frozen constrained beam", test_gold, test_predictions)
])

,model,token_accuracy,entity_token_macro_f1,span_precision,span_recall,span_f1
0,Frozen constrained beam,0.940304,0.713705,0.680026,0.733003,0.705521


# What later architectures change

RNNs and Transformers can replace hand-built context features with learned contextual representations. A CRF can learn transition scores. The BIO scheme, alignment checks, baselines, decoding questions and span-level evaluation remain the same.

# Checklist

Before leaving the lab, verify that you can:

- explain why `O` accuracy is an insufficient NER metric;
- identify an invalid BIO transition;
- distinguish independent scoring from constrained decoding;
- explain what beam size controls; and
- round-trip a character answer span through token-level BIO labels.